# **New York City Yellow Taxi Data**

## Objective
In this case study you will be learning exploratory data analysis (EDA) with the help of a dataset on yellow taxi rides in New York City. This will enable you to understand why EDA is an important step in the process of data science and machine learning.

## **Problem Statement**
As an analyst at an upcoming taxi operation in NYC, you are tasked to use the 2023 taxi trip data to uncover insights that could help optimise taxi operations. The goal is to analyse patterns in the data that can inform strategic decisions to improve service efficiency, maximise revenue, and enhance passenger experience.

## Tasks
You need to perform the following steps for successfully completing this assignment:
1. Data Loading
2. Data Cleaning
3. Exploratory Analysis: Bivariate and Multivariate
4. Creating Visualisations to Support the Analysis
5. Deriving Insights and Stating Conclusions

---

**NOTE:** The marks given along with headings and sub-headings are cumulative marks for those particular headings/sub-headings.<br>

The actual marks for each task are specified within the tasks themselves.

For example, marks given with heading *2* or sub-heading *2.1* are the cumulative marks, for your reference only. <br>

The marks you will receive for completing tasks are given with the tasks.

Suppose the marks for two tasks are: 3 marks for 2.1.1 and 2 marks for 3.2.2, or
* 2.1.1 [3 marks]
* 3.2.2 [2 marks]

then, you will earn 3 marks for completing task 2.1.1 and 2 marks for completing task 3.2.2.


---

## Data Understanding
The yellow taxi trip records include fields capturing pick-up and drop-off dates/times, pick-up and drop-off locations, trip distances, itemized fares, rate types, payment types, and driver-reported passenger counts.

The data is stored in Parquet format (*.parquet*). The dataset is from 2009 to 2024. However, for this assignment, we will only be using the data from 2023.

The data for each month is present in a different parquet file. You will get twelve files for each of the months in 2023.

The data was collected and provided to the NYC Taxi and Limousine Commission (TLC) by technology providers like vendors and taxi hailing apps. <br>

You can find the link to the TLC trip records page here: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

###  Data Description
You can find the data description here: [Data Dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf)

**Trip Records**



|Field Name       |description |
|:----------------|:-----------|
| VendorID | A code indicating the TPEP provider that provided the record. <br> 1= Creative Mobile Technologies, LLC; <br> 2= VeriFone Inc. |
| tpep_pickup_datetime | The date and time when the meter was engaged.  |
| tpep_dropoff_datetime | The date and time when the meter was disengaged.   |
| Passenger_count | The number of passengers in the vehicle. <br> This is a driver-entered value. |
| Trip_distance | The elapsed trip distance in miles reported by the taximeter. |
| PULocationID | TLC Taxi Zone in which the taximeter was engaged |
| DOLocationID | TLC Taxi Zone in which the taximeter was disengaged |
|RateCodeID |The final rate code in effect at the end of the trip.<br> 1 = Standard rate <br> 2 = JFK <br> 3 = Newark <br>4 = Nassau or Westchester <br>5 = Negotiated fare <br>6 = Group ride |
|Store_and_fwd_flag |This flag indicates whether the trip record was held in vehicle memory before sending to the vendor, aka “store and forward,” because the vehicle did not have a connection to the server.  <br>Y= store and forward trip <br>N= not a store and forward trip |
|Payment_type| A numeric code signifying how the passenger paid for the trip. <br> 1 = Credit card <br>2 = Cash <br>3 = No charge <br>4 = Dispute <br>5 = Unknown <br>6 = Voided trip |
|Fare_amount| The time-and-distance fare calculated by the meter. <br>Extra Miscellaneous extras and surcharges.  Currently, this only includes the 0.50 and 1 USD rush hour and overnight charges. |
|MTA_tax |0.50 USD MTA tax that is automatically triggered based on the metered rate in use. |
|Improvement_surcharge | 0.30 USD improvement surcharge assessed trips at the flag drop. The improvement surcharge began being levied in 2015. |
|Tip_amount |Tip amount – This field is automatically populated for credit card tips. Cash tips are not included. |
| Tolls_amount | Total amount of all tolls paid in trip.  |
| total_amount | The total amount charged to passengers. Does not include cash tips. |
|Congestion_Surcharge |Total amount collected in trip for NYS congestion surcharge. |
| Airport_fee | 1.25 USD for pick up only at LaGuardia and John F. Kennedy Airports|

Although the amounts of extra charges and taxes applied are specified in the data dictionary, you will see that some cases have different values of these charges in the actual data.

**Taxi Zones**

Each of the trip records contains a field corresponding to the location of the pickup or drop-off of the trip, populated by numbers ranging from 1-263.

These numbers correspond to taxi zones, which may be downloaded as a table or map/shapefile and matched to the trip records using a join.

This is covered in more detail in later sections.

---

## **1** Data Preparation

<font color = red>[5 marks]</font> <br>

### Import Libraries

In [ ]:
import warnings

warnings.filterwarnings("ignore")


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("Set2")
pd.set_option("display.max_columns", 100)

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "Datasets and Dictionary"
TRIP_RECORDS_DIR = DATA_DIR / "trip_records"
TAXI_ZONES_PATH = DATA_DIR / "taxi_zones" / "taxi_zones.shp"
SAMPLED_PARQUET_PATH = PROJECT_ROOT / "yellow_taxi_2023_sampled.parquet"
SAMPLED_CSV_PATH = PROJECT_ROOT / "yellow_taxi_2023_sampled.csv"
SAMPLE_FRACTION = 0.008
RANDOM_STATE = 42
NIGHT_HOURS = [23, 0, 1, 2, 3, 4, 5]
DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
MONTH_ORDER = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]
PAYMENT_TYPE_MAP = {
    1: "Credit card",
    2: "Cash",
    3: "No charge",
    4: "Dispute",
    5: "Unknown",
    6: "Voided trip",
}
VENDOR_MAP = {
    1: "Creative Mobile Technologies",
    2: "VeriFone",
}
RATECODE_MAP = {
    1: "Standard rate",
    2: "JFK",
    3: "Newark",
    4: "Nassau/Westchester",
    5: "Negotiated fare",
    6: "Group ride",
}


def get_trip_files():
    return sorted(TRIP_RECORDS_DIR.glob("*.parquet"), key=lambda path: int(path.stem.split("-")[-1]))


def resolve_column(dataframe, candidate):
    matches = [column for column in dataframe.columns if column.lower() == candidate.lower()]
    if not matches:
        raise KeyError(f"Column '{candidate}' not found.")
    return matches[0]


def sample_by_date_hour(month_df, pickup_col, frac=SAMPLE_FRACTION, random_state=RANDOM_STATE):
    temp = month_df.copy()
    temp[pickup_col] = pd.to_datetime(temp[pickup_col], errors="coerce")
    temp = temp.dropna(subset=[pickup_col]).copy()
    temp["_pickup_date"] = temp[pickup_col].dt.date
    temp["_pickup_hour"] = temp[pickup_col].dt.hour

    def take_sample(group):
        sample_size = max(1, int(np.ceil(len(group) * frac)))
        sample_size = min(sample_size, len(group))
        return group.sample(n=sample_size, random_state=random_state)

    sampled = (
        temp.groupby(["_pickup_date", "_pickup_hour"], group_keys=False, sort=False)
        .apply(take_sample)
        .drop(columns=["_pickup_date", "_pickup_hour"], errors="ignore")
        .reset_index(drop=True)
    )
    return sampled


def add_time_features(dataframe):
    enriched = dataframe.copy()
    enriched["tpep_pickup_datetime"] = pd.to_datetime(enriched["tpep_pickup_datetime"], errors="coerce")
    enriched["tpep_dropoff_datetime"] = pd.to_datetime(enriched["tpep_dropoff_datetime"], errors="coerce")
    enriched["pickup_date"] = enriched["tpep_pickup_datetime"].dt.date
    enriched["pickup_hour"] = enriched["tpep_pickup_datetime"].dt.hour
    enriched["pickup_day_name"] = pd.Categorical(
        enriched["tpep_pickup_datetime"].dt.day_name(),
        categories=DAY_ORDER,
        ordered=True,
    )
    enriched["pickup_day_num"] = enriched["tpep_pickup_datetime"].dt.dayofweek
    enriched["pickup_month"] = enriched["tpep_pickup_datetime"].dt.month
    enriched["pickup_month_name"] = pd.Categorical(
        enriched["tpep_pickup_datetime"].dt.month_name(),
        categories=MONTH_ORDER,
        ordered=True,
    )
    enriched["pickup_quarter"] = enriched["tpep_pickup_datetime"].dt.quarter
    enriched["is_weekend"] = enriched["pickup_day_num"] >= 5
    enriched["trip_duration_min"] = (
        enriched["tpep_dropoff_datetime"] - enriched["tpep_pickup_datetime"]
    ).dt.total_seconds() / 60
    enriched["trip_duration_hr"] = enriched["trip_duration_min"] / 60
    return enriched


In [ ]:
print("numpy version:", np.__version__)
print("pandas version:", pd.__version__)
print("matplotlib version:", plt.matplotlib.__version__)
print("seaborn version:", sns.__version__)
print("Trip record files found:", len(get_trip_files()))
print("Taxi zones shapefile available:", TAXI_ZONES_PATH.exists())


### **1.1** Load the dataset
<font color = red>[5 marks]</font> <br>

You will see twelve files, one for each month.

To read parquet files with Pandas, you have to follow a similar syntax as that for CSV files.

`df = pd.read_parquet('file.parquet')`

In [ ]:
trip_files = get_trip_files()
example_df = pd.read_parquet(trip_files[0])
print(f"Loaded {trip_files[0].name}")
print(f"Rows: {len(example_df):,} | Columns: {len(example_df.columns)}")
display(example_df.head())
example_df.info()


How many rows are there? Do you think handling such a large number of rows is computationally feasible when we have to combine the data for all twelve months into one?

To handle this, we need to sample a fraction of data from each of the files. How to go about that? Think of a way to select only some portion of the data from each month's file that accurately represents the trends.

#### Sampling the Data
> One way is to take a small percentage of entries for pickup in every hour of a date. So, for all the days in a month, we can iterate through the hours and select 5% values randomly from those. Use `tpep_pickup_datetime` for this. Separate date and hour from the datetime values and then for each date, select some fraction of trips for each of the 24 hours.

To sample data, you can use the `sample()` method. Follow this syntax:

```Python
# sampled_data is an empty DF to keep appending sampled data of each hour
# hour_data is the DF of entries for an hour 'X' on a date 'Y'

sample = hour_data.sample(frac = 0.05, random_state = 42)
# sample 0.05 of the hour_data
# random_state is just a seed for sampling, you can define it yourself

sampled_data = pd.concat([sampled_data, sample]) # adding data for this hour to the DF
```

This *sampled_data* will contain 5% values selected at random from each hour.

Note that the code given above is only the part that will be used for sampling and not the complete code required for sampling and combining the data files.

Keep in mind that you sample by date AND hour, not just hour. (Why?)

---

**1.1.1** <font color = red>[5 marks]</font> <br>
Figure out how to sample and combine the files.

**Note:** It is not mandatory to use the method specified above. While sampling, you only need to make sure that your sampled data represents the overall data of all the months accurately.

In [ ]:
trip_files = get_trip_files()
monthly_samples = []
sampling_rows = []

for file_path in trip_files:
    month_df = pd.read_parquet(file_path)
    pickup_col = resolve_column(month_df, "tpep_pickup_datetime")
    sampled_month = sample_by_date_hour(month_df, pickup_col)
    monthly_samples.append(sampled_month)
    sampling_rows.append(
        {
            "file": file_path.name,
            "original_rows": len(month_df),
            "sampled_rows": len(sampled_month),
            "sample_ratio": round(len(sampled_month) / len(month_df), 4),
        }
    )
    print(
        f"{file_path.name}: {len(month_df):,} rows -> {len(sampled_month):,} sampled rows"
    )

df = pd.concat(monthly_samples, ignore_index=True)
sampling_summary = pd.DataFrame(sampling_rows)
print(f"Combined sampled dataset shape: {df.shape}")
display(sampling_summary)


In [ ]:
# The repository already contains the data files, so Google Drive mounting is not required.


In [ ]:
print(f"Sampling fraction used: {SAMPLE_FRACTION:.3%}")
print(f"Total sampled rows across all months: {len(df):,}")
print(f"Average monthly sample size: {sampling_summary['sampled_rows'].mean():,.0f}")


After combining the data files into one DataFrame, convert the new DataFrame to a CSV or parquet file and store it to use directly.

Ideally, you can try keeping the total entries to around 250,000 to 300,000.

In [ ]:
sampled_output_path = None

try:
    df.to_parquet(SAMPLED_PARQUET_PATH, index=False)
    sampled_output_path = SAMPLED_PARQUET_PATH
except Exception as error:
    print(f"Parquet save failed: {error}")
    df.to_csv(SAMPLED_CSV_PATH, index=False)
    sampled_output_path = SAMPLED_CSV_PATH

print(f"Sampled dataset saved to: {sampled_output_path}")


## **2** Data Cleaning
<font color = red>[30 marks]</font> <br>

Now we can load the new data directly.

In [ ]:
if SAMPLED_PARQUET_PATH.exists():
    sampled_output_path = SAMPLED_PARQUET_PATH
    df = pd.read_parquet(sampled_output_path)
else:
    sampled_output_path = SAMPLED_CSV_PATH
    df = pd.read_csv(sampled_output_path)

print(f"Loaded sampled dataset from: {sampled_output_path}")
print(f"Shape: {df.shape}")


In [ ]:
display(df.head())


In [ ]:
df.info()


#### **2.1** Fixing Columns
<font color = red>[10 marks]</font> <br>

Fix/drop any columns as you seem necessary in the below sections

**2.1.1** <font color = red>[2 marks]</font> <br>

Fix the index and drop unnecessary columns

In [ ]:
df = df.reset_index(drop=True)
drop_columns = [
    column
    for column in df.columns
    if column.lower().startswith("unnamed") or column.lower() in {"index", "level_0"}
]
if drop_columns:
    df = df.drop(columns=drop_columns)

print(f"Shape after index reset and cleanup: {df.shape}")
print("Columns retained:")
display(pd.Series(df.columns, name="column_name"))


**2.1.2** <font color = red>[3 marks]</font> <br>
There are two airport fee columns. This is possibly an error in naming columns. Let's see whether these can be combined into a single column.

In [ ]:
airport_fee_columns = [column for column in df.columns if column.lower() == "airport_fee"]

if len(airport_fee_columns) > 1:
    airport_fee_values = pd.concat(
        [pd.to_numeric(df[column], errors="coerce") for column in airport_fee_columns],
        axis=1,
    )
    df = df.drop(columns=airport_fee_columns)
    df["airport_fee"] = airport_fee_values.fillna(0).max(axis=1)
elif len(airport_fee_columns) == 1 and airport_fee_columns[0] != "airport_fee":
    df = df.rename(columns={airport_fee_columns[0]: "airport_fee"})
elif len(airport_fee_columns) == 0:
    df["airport_fee"] = 0.0

print("Airport-fee related columns after cleanup:")
display([column for column in df.columns if "airport" in column.lower()])
display(df[[column for column in df.columns if "airport" in column.lower()]].head())


**2.1.3** <font color = red>[5 marks]</font> <br>
Fix columns with negative (monetary) values

In [ ]:
fare_col = resolve_column(df, "fare_amount")
ratecode_col = resolve_column(df, "RatecodeID")
selected_columns = [
    column
    for column in [fare_col, ratecode_col, "total_amount", "tip_amount", "airport_fee"]
    if column in df.columns
]
negative_fare_mask = pd.to_numeric(df[fare_col], errors="coerce") < 0
negative_fare_rows = df.loc[negative_fare_mask, selected_columns].copy()

print(f"Rows with negative fare_amount: {negative_fare_mask.sum():,}")
display(negative_fare_rows.head())


Did you notice something different in the `RatecodeID` column for above records?

In [ ]:
negative_ratecode_distribution = (
    df.loc[negative_fare_mask, ratecode_col]
    .value_counts(dropna=False)
    .rename_axis("RatecodeID")
    .reset_index(name="count")
)

display(negative_ratecode_distribution)


In [ ]:
monetary_columns = []
for candidate in [
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
]:
    try:
        monetary_columns.append(resolve_column(df, candidate))
    except KeyError:
        continue

negative_value_summary = pd.DataFrame(
    {
        "column": monetary_columns,
        "negative_values": [
            int((pd.to_numeric(df[column], errors="coerce") < 0).sum())
            for column in monetary_columns
        ],
    }
).sort_values("negative_values", ascending=False)

display(negative_value_summary)


In [ ]:
valid_ratecodes = set(RATECODE_MAP)
negative_value_mask = pd.DataFrame(
    {
        column: pd.to_numeric(df[column], errors="coerce") < 0
        for column in monetary_columns
    }
).any(axis=1)
ratecode_series = pd.to_numeric(df[ratecode_col], errors="coerce")
invalid_negative_rows = negative_value_mask & ~ratecode_series.isin(valid_ratecodes)

rows_removed = int(invalid_negative_rows.sum())
df = df.loc[~invalid_negative_rows].copy()

for column in monetary_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")
    df[column] = df[column].mask(df[column] < 0, 0)

print(f"Dropped {rows_removed:,} rows where negative monetary values appeared with invalid RatecodeID values.")
print("Remaining negative values by monetary column:")
display(
    pd.Series(
        {column: int((df[column] < 0).sum()) for column in monetary_columns},
        name="remaining_negative_values",
    )
)


### **2.2** Handling Missing Values
<font color = red>[10 marks]</font> <br>

**2.2.1**  <font color = red>[2 marks]</font> <br>
Find the proportion of missing values in each column




In [ ]:
missing_proportion = (
    df.isna().mean().mul(100).sort_values(ascending=False).rename("missing_percent")
)
missing_proportion = missing_proportion[missing_proportion > 0]

display(missing_proportion.to_frame())


**2.2.2**  <font color = red>[3 marks]</font> <br>
Handling missing values in `passenger_count`

In [ ]:
passenger_col = resolve_column(df, "passenger_count")
pickup_col = resolve_column(df, "tpep_pickup_datetime")
passenger_series = pd.to_numeric(df[passenger_col], errors="coerce")
null_or_zero_passenger_mask = passenger_series.isna() | (passenger_series == 0)

print(f"Rows with missing or zero passenger_count: {null_or_zero_passenger_mask.sum():,}")
display(df.loc[null_or_zero_passenger_mask, [pickup_col, passenger_col]].head())

mode_passenger_count = int(passenger_series[passenger_series > 0].mode().iloc[0])
df[passenger_col] = passenger_series.replace(0, np.nan).fillna(mode_passenger_count).round().astype(int)
print(f"Filled missing/zero passenger_count values with mode: {mode_passenger_count}")


Did you find zeroes in passenger_count? Handle these.

**2.2.3**  <font color = red>[2 marks]</font> <br>
Handle missing values in `RatecodeID`

In [ ]:
ratecode_col = resolve_column(df, "RatecodeID")
ratecode_series = pd.to_numeric(df[ratecode_col], errors="coerce")
mode_ratecode = int(ratecode_series.dropna().mode().iloc[0])
df[ratecode_col] = ratecode_series.fillna(mode_ratecode).round().astype(int)

print(f"Filled missing RatecodeID values with mode: {mode_ratecode}")
display(df[ratecode_col].value_counts().sort_index())


**2.2.4**  <font color = red>[3 marks]</font> <br>
Impute NaN in `congestion_surcharge`

In [ ]:
try:
    congestion_col = resolve_column(df, "congestion_surcharge")
    missing_before = int(df[congestion_col].isna().sum())
    df[congestion_col] = pd.to_numeric(df[congestion_col], errors="coerce").fillna(0)
    print(f"Filled {missing_before:,} missing congestion_surcharge values with 0.")
except KeyError:
    print("congestion_surcharge column not found in the sampled dataset.")


Are there missing values in other columns? Did you find NaN values in some other set of columns? Handle those missing values below.

In [ ]:
critical_columns = []
for candidate in [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount",
    "total_amount",
]:
    try:
        critical_columns.append(resolve_column(df, candidate))
    except KeyError:
        continue

rows_before_missing_cleanup = len(df)
df = df.dropna(subset=critical_columns).copy()
rows_after_missing_cleanup = len(df)

for candidate in ["VendorID", "payment_type"]:
    try:
        column = resolve_column(df, candidate)
        mode_value = pd.to_numeric(df[column], errors="coerce").dropna().mode().iloc[0]
        df[column] = pd.to_numeric(df[column], errors="coerce").fillna(mode_value)
    except KeyError:
        continue

try:
    store_flag_col = resolve_column(df, "store_and_fwd_flag")
    df[store_flag_col] = df[store_flag_col].fillna("N")
except KeyError:
    pass

for candidate in [
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "airport_fee",
]:
    try:
        column = resolve_column(df, candidate)
        df[column] = pd.to_numeric(df[column], errors="coerce").fillna(0)
    except KeyError:
        continue

print(
    f"Dropped {rows_before_missing_cleanup - rows_after_missing_cleanup:,} rows with missing values in critical trip columns."
)
print("Top remaining missing-value percentages:")
display((df.isna().mean().mul(100).sort_values(ascending=False).head(10)).to_frame("missing_percent"))


### **2.3** Handling Outliers
<font color = red>[10 marks]</font> <br>

Before we start fixing outliers, let's perform outlier analysis.

In [ ]:
outlier_columns = []
for candidate in [
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "passenger_count",
    "airport_fee",
]:
    try:
        column = resolve_column(df, candidate)
        df[column] = pd.to_numeric(df[column], errors="coerce")
        outlier_columns.append(column)
    except KeyError:
        continue

outlier_summary = df[outlier_columns].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T
display(outlier_summary)


**2.3.1**  <font color = red>[10 marks]</font> <br>
Based on the above analysis, it seems that some of the outliers are present due to errors in registering the trips. Fix the outliers.

Some points you can look for:
- Entries where `trip_distance` is nearly 0 and `fare_amount` is more than 300
- Entries where `trip_distance` and `fare_amount` are 0 but the pickup and dropoff zones are different (both distance and fare should not be zero for different zones)
- Entries where `trip_distance` is more than 250  miles.
- Entries where `payment_type` is 0 (there is no payment_type 0 defined in the data dictionary)

These are just some suggestions. You can handle outliers in any way you wish, using the insights from above outlier analysis.

How will you fix each of these values? Which ones will you drop and which ones will you replace?

First, let us remove 7+ passenger counts as there are very less instances.

In [ ]:
passenger_col = resolve_column(df, "passenger_count")
rows_before_passenger_filter = len(df)
df = df[pd.to_numeric(df[passenger_col], errors="coerce").between(1, 6, inclusive="both")].copy()

print(
    f"Removed {rows_before_passenger_filter - len(df):,} rows where passenger_count was outside the 1 to 6 range."
)


In [ ]:
pickup_col = resolve_column(df, "tpep_pickup_datetime")
dropoff_col = resolve_column(df, "tpep_dropoff_datetime")

for datetime_col in [pickup_col, dropoff_col]:
    df[datetime_col] = pd.to_datetime(df[datetime_col], errors="coerce")

df["trip_duration_min_temp"] = (
    df[dropoff_col] - df[pickup_col]
).dt.total_seconds() / 60

df = df[df["trip_duration_min_temp"].between(1, 240, inclusive="both")].copy()

for candidate in ["trip_distance", "fare_amount", "tip_amount", "total_amount"]:
    column = resolve_column(df, candidate)
    upper_limit = df[column].quantile(0.995)
    df[column] = df[column].clip(lower=0, upper=upper_limit)

duration_upper_limit = df["trip_duration_min_temp"].quantile(0.995)
df["trip_duration_min_temp"] = df["trip_duration_min_temp"].clip(upper=duration_upper_limit)

print(f"Shape after outlier treatment: {df.shape}")
display(
    df[[
        resolve_column(df, "trip_distance"),
        resolve_column(df, "fare_amount"),
        resolve_column(df, "tip_amount"),
        resolve_column(df, "total_amount"),
        "trip_duration_min_temp",
    ]].describe(percentiles=[0.5, 0.95, 0.99]).T
)


In [ ]:
df.columns = [column.strip().lower() for column in df.columns]
df = df.loc[:, ~df.columns.duplicated()].copy()

for integer_column in ["vendorid", "passenger_count", "ratecodeid", "pulocationid", "dolocationid", "payment_type"]:
    if integer_column in df.columns:
        df[integer_column] = pd.to_numeric(df[integer_column], errors="coerce").fillna(0).round().astype(int)

if "ratecodeid" in df.columns:
    df.loc[~df["ratecodeid"].isin(RATECODE_MAP), "ratecodeid"] = 1

if "store_and_fwd_flag" in df.columns:
    df["store_and_fwd_flag"] = df["store_and_fwd_flag"].astype(str).str.upper().str.strip().replace({"NAN": "N"})

if "airport_fee" not in df.columns:
    df["airport_fee"] = 0.0

if "trip_duration_min_temp" in df.columns:
    df = df.drop(columns=["trip_duration_min_temp"])

df = add_time_features(df)
df = df.dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime"]).copy()
df = df[df["trip_duration_min"] > 0].copy()

df["vendor_name"] = df["vendorid"].map(VENDOR_MAP).fillna("Unknown vendor")
df["payment_type_name"] = df["payment_type"].map(PAYMENT_TYPE_MAP).fillna("Unknown")
df["ratecode_name"] = df["ratecodeid"].map(RATECODE_MAP).fillna("Other")

print(f"Final cleaned dataset shape: {df.shape}")
print("Cleaned column names:")
display(df.columns.tolist())


## **3** Exploratory Data Analysis
<font color = red>[90 marks]</font> <br>

In [ ]:
numerical_columns = [
    column for column in [
        "passenger_count",
        "trip_distance",
        "fare_amount",
        "extra",
        "mta_tax",
        "tip_amount",
        "tolls_amount",
        "improvement_surcharge",
        "total_amount",
        "congestion_surcharge",
        "airport_fee",
        "trip_duration_min",
    ]
    if column in df.columns
]

categorical_columns = [
    column for column in [
        "vendorid",
        "vendor_name",
        "ratecodeid",
        "ratecode_name",
        "store_and_fwd_flag",
        "pulocationid",
        "dolocationid",
        "payment_type",
        "payment_type_name",
        "pickup_day_name",
        "pickup_month_name",
    ]
    if column in df.columns
]

datetime_columns = [
    column for column in ["tpep_pickup_datetime", "tpep_dropoff_datetime"]
    if column in df.columns
]

print("Numerical columns:")
display(numerical_columns)
print("Categorical columns:")
display(categorical_columns)
print("Datetime columns:")
display(datetime_columns)


#### **3.1** General EDA: Finding Patterns and Trends
<font color = red>[40 marks]</font> <br>

**3.1.1** <font color = red>[3 marks]</font> <br>
Categorise the varaibles into Numerical or Categorical.
* `VendorID`:
* `tpep_pickup_datetime`:
* `tpep_dropoff_datetime`:
* `passenger_count`:
* `trip_distance`:
* `RatecodeID`:
* `PULocationID`:
* `DOLocationID`:
* `payment_type`:
* `pickup_hour`:
* `trip_duration`:


The following monetary parameters belong in the same category, is it categorical or numerical?


* `fare_amount`
* `extra`
* `mta_tax`
* `tip_amount`
* `tolls_amount`
* `improvement_surcharge`
* `total_amount`
* `congestion_surcharge`
* `airport_fee`

##### Temporal Analysis

**3.1.2** <font color = red>[5 marks]</font> <br>
Analyse the distribution of taxi pickups by hours, days of the week, and months.

In [ ]:
hourly_pickups = df.groupby("pickup_hour").size().reindex(range(24), fill_value=0)
general_busiest_hour = int(hourly_pickups.idxmax())

plt.figure(figsize=(12, 4))
sns.barplot(x=hourly_pickups.index, y=hourly_pickups.values, color="#4C72B0")
plt.title("Taxi Pickups by Hour of Day")
plt.xlabel("Pickup hour")
plt.ylabel("Number of sampled trips")
plt.xticks(range(24))
plt.show()

print(f"Busiest sampled hour: {general_busiest_hour}:00 with {hourly_pickups.max():,} trips")


In [ ]:
daily_pickups = (
    df.groupby("pickup_day_name", observed=False)
    .size()
    .reindex(DAY_ORDER)
    .fillna(0)
)
general_busiest_day = daily_pickups.idxmax()

plt.figure(figsize=(10, 4))
sns.barplot(x=daily_pickups.index, y=daily_pickups.values, palette="crest")
plt.title("Taxi Pickups by Day of Week")
plt.xlabel("Day of week")
plt.ylabel("Number of sampled trips")
plt.xticks(rotation=30)
plt.show()

print(f"Busiest sampled day: {general_busiest_day}")


In [ ]:
monthly_pickups = (
    df.groupby("pickup_month_name", observed=False)
    .size()
    .reindex(MONTH_ORDER)
    .fillna(0)
)
general_busiest_month = monthly_pickups.idxmax()

plt.figure(figsize=(12, 4))
sns.lineplot(x=monthly_pickups.index, y=monthly_pickups.values, marker="o", linewidth=2)
plt.title("Taxi Pickups by Month")
plt.xlabel("Month")
plt.ylabel("Number of sampled trips")
plt.xticks(rotation=30)
plt.show()

print(f"Busiest sampled month: {general_busiest_month}")


##### Financial Analysis

Take a look at the financial parameters like `fare_amount`, `tip_amount`, `total_amount`, and also `trip_distance`. Do these contain zero/negative values?

In [ ]:
financial_quality_summary = pd.DataFrame(
    {
        "column": ["fare_amount", "tip_amount", "total_amount", "trip_distance"],
        "zero_values": [
            int((df[column] == 0).sum())
            for column in ["fare_amount", "tip_amount", "total_amount", "trip_distance"]
        ],
        "negative_values": [
            int((df[column] < 0).sum())
            for column in ["fare_amount", "tip_amount", "total_amount", "trip_distance"]
        ],
        "minimum": [df[column].min() for column in ["fare_amount", "tip_amount", "total_amount", "trip_distance"]],
        "maximum": [df[column].max() for column in ["fare_amount", "tip_amount", "total_amount", "trip_distance"]],
    }
)

display(financial_quality_summary)


Do you think it is beneficial to create a copy DataFrame leaving out the zero values from these?

**3.1.3** <font color = red>[2 marks]</font> <br>
Filter out the zero values from the above columns.

**Note:** The distance might be 0 in cases where pickup and drop is in the same zone. Do you think it is suitable to drop such cases of zero distance?

In [ ]:
revenue_df = df[
    (df["fare_amount"] > 0)
    & (df["total_amount"] > 0)
    & (df["trip_distance"] > 0)
].copy()

tip_analysis_df = revenue_df[revenue_df["tip_amount"] > 0].copy()

print(f"Rows retained for revenue and fare analysis: {len(revenue_df):,}")
print(f"Rows retained for tip-based analysis: {len(tip_analysis_df):,}")


**3.1.4** <font color = red>[3 marks]</font> <br>
Analyse the monthly revenue (`total_amount`) trend

In [ ]:
monthly_revenue = (
    revenue_df.groupby("pickup_month_name", observed=False)["total_amount"]
    .sum()
    .reindex(MONTH_ORDER)
)

plt.figure(figsize=(12, 4))
sns.lineplot(x=monthly_revenue.index, y=monthly_revenue.values, marker="o", linewidth=2)
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Total sampled revenue")
plt.xticks(rotation=30)
plt.show()

display(monthly_revenue.to_frame("total_revenue"))


**3.1.5** <font color = red>[3 marks]</font> <br>
Show the proportion of each quarter of the year in the revenue

In [ ]:
quarterly_revenue = revenue_df.groupby("pickup_quarter")["total_amount"].sum().sort_index()
quarterly_revenue_share = (quarterly_revenue / quarterly_revenue.sum() * 100).round(2)

plt.figure(figsize=(8, 4))
sns.barplot(x=quarterly_revenue_share.index, y=quarterly_revenue_share.values, color="#55A868")
plt.title("Quarterly Share of Yearly Revenue")
plt.xlabel("Quarter")
plt.ylabel("Revenue share (%)")
plt.show()

display(quarterly_revenue_share.to_frame("revenue_share_percent"))


**3.1.6** <font color = red>[3 marks]</font> <br>
Visualise the relationship between `trip_distance` and `fare_amount`. Also find the correlation value for these two.

**Hint:** You can leave out the trips with trip_distance = 0

In [ ]:
distance_fare_corr = revenue_df[["trip_distance", "fare_amount"]].corr().loc["trip_distance", "fare_amount"]
scatter_distance_fare = revenue_df.sample(min(len(revenue_df), 10000), random_state=RANDOM_STATE)

plt.figure(figsize=(8, 5))
sns.regplot(
    data=scatter_distance_fare,
    x="trip_distance",
    y="fare_amount",
    scatter_kws={"alpha": 0.2, "s": 10},
    line_kws={"color": "red"},
)
plt.title("Fare Amount vs Trip Distance")
plt.xlabel("Trip distance (miles)")
plt.ylabel("Fare amount (USD)")
plt.show()

print(f"Correlation between trip_distance and fare_amount: {distance_fare_corr:.3f}")


**3.1.7** <font color = red>[5 marks]</font> <br>
Find and visualise the correlation between:
1. `fare_amount` and trip duration (pickup time to dropoff time)
2. `fare_amount` and `passenger_count`
3. `tip_amount` and `trip_distance`

In [ ]:
duration_fare_corr = revenue_df[["trip_duration_min", "fare_amount"]].corr().loc["trip_duration_min", "fare_amount"]
scatter_duration_fare = revenue_df.sample(min(len(revenue_df), 10000), random_state=RANDOM_STATE)

plt.figure(figsize=(8, 5))
sns.regplot(
    data=scatter_duration_fare,
    x="trip_duration_min",
    y="fare_amount",
    scatter_kws={"alpha": 0.2, "s": 10},
    line_kws={"color": "red"},
)
plt.title("Fare Amount vs Trip Duration")
plt.xlabel("Trip duration (minutes)")
plt.ylabel("Fare amount (USD)")
plt.show()

print(f"Correlation between trip duration and fare_amount: {duration_fare_corr:.3f}")


In [ ]:
fare_by_passenger_count = revenue_df.groupby("passenger_count")["fare_amount"].mean().sort_index()

plt.figure(figsize=(8, 4))
sns.barplot(x=fare_by_passenger_count.index, y=fare_by_passenger_count.values, color="#C44E52")
plt.title("Average Fare by Passenger Count")
plt.xlabel("Passenger count")
plt.ylabel("Average fare amount (USD)")
plt.show()

display(fare_by_passenger_count.to_frame("avg_fare_amount"))


In [ ]:
tip_distance_df = tip_analysis_df.copy()
tip_distance_df["distance_band"] = pd.cut(
    tip_distance_df["trip_distance"],
    bins=[0, 1, 2, 5, 10, np.inf],
    labels=["0-1", "1-2", "2-5", "5-10", "10+"],
    include_lowest=True,
)

tip_by_distance = tip_distance_df.groupby("distance_band", observed=False)["tip_amount"].mean()

plt.figure(figsize=(8, 4))
sns.barplot(x=tip_by_distance.index, y=tip_by_distance.values, color="#8172B2")
plt.title("Average Tip Amount by Trip Distance Band")
plt.xlabel("Trip distance band (miles)")
plt.ylabel("Average tip amount (USD)")
plt.show()

display(tip_by_distance.to_frame("avg_tip_amount"))


**3.1.8** <font color = red>[3 marks]</font> <br>
Analyse the distribution of different payment types (`payment_type`)

In [ ]:
payment_distribution = df["payment_type_name"].value_counts()

plt.figure(figsize=(8, 4))
sns.barplot(x=payment_distribution.index, y=payment_distribution.values, palette="viridis")
plt.title("Distribution of Payment Types")
plt.xlabel("Payment type")
plt.ylabel("Number of sampled trips")
plt.xticks(rotation=30)
plt.show()

display(payment_distribution.to_frame("trip_count"))


- 1= Credit card
- 2= Cash
- 3= No charge
- 4= Dispute



##### Geographical Analysis

For this, you have to use the *taxi_zones.shp* file from the *taxi_zones* folder.

There would be multiple files inside the folder (such as *.shx, .sbx, .sbn* etc). You do not need to import/read any of the files other than the shapefile, *taxi_zones.shp*.

Do not change any folder structure - all the files need to be present inside the folder for it to work.

The folder structure should look like this:
```
Taxi Zones
|- taxi_zones.shp.xml
|- taxi_zones.prj
|- taxi_zones.sbn
|- taxi_zones.shp
|- taxi_zones.dbf
|- taxi_zones.shx
|- taxi_zones.sbx

 ```

 You only need to read the `taxi_zones.shp` file. The *shp* file will utilise the other files by itself.

We will use the *GeoPandas* library for geopgraphical analysis
```
import geopandas as gpd
```

More about geopandas and shapefiles: [About](https://geopandas.org/en/stable/about.html)


Reading the shapefile is very similar to *Pandas*. Use `gpd.read_file()` function to load the data (*taxi_zones.shp*) as a GeoDataFrame. Documentation: [Reading and Writing Files](https://geopandas.org/en/stable/docs/user_guide/io.html)

In [ ]:
# If GeoPandas is unavailable in your notebook environment, install it before running the next cells.
# Example: !pip install geopandas


**3.1.9** <font color = red>[2 marks]</font> <br>
Load the shapefile and display it.

In [ ]:
import geopandas as gpd

zones = gpd.read_file(TAXI_ZONES_PATH)
zones.columns = [column.strip().lower() for column in zones.columns]
print(f"Zones dataframe shape: {zones.shape}")
display(zones.head())


Now, if you look at the DataFrame created, you will see columns like: `OBJECTID`,`Shape_Leng`, `Shape_Area`, `zone`, `LocationID`, `borough`, `geometry`.
<br><br>

Now, the `locationID` here is also what we are using to mark pickup and drop zones in the trip records.

The geometric parameters like shape length, shape area and geometry are used to plot the zones on a map.

This can be easily done using the `plot()` method.

In [ ]:
print(zones.info())

ax = zones.plot(figsize=(12, 10), edgecolor="black", linewidth=0.2, color="#F2F2F2")
ax.set_title("NYC Taxi Zones")
ax.set_axis_off()
plt.show()


Now, you have to merge the trip records and zones data using the location IDs.



**3.1.10** <font color = red>[3 marks]</font> <br>
Merge the zones data into trip data using the `locationID` and `PULocationID` columns.

In [ ]:
zone_lookup = zones[["locationid", "zone", "borough"]].copy()

trip_zones = df.merge(
    zone_lookup.add_prefix("pickup_"),
    left_on="pulocationid",
    right_on="pickup_locationid",
    how="left",
)
trip_zones = trip_zones.merge(
    zone_lookup.add_prefix("dropoff_"),
    left_on="dolocationid",
    right_on="dropoff_locationid",
    how="left",
)

display(
    trip_zones[
        [
            "pulocationid",
            "pickup_zone",
            "pickup_borough",
            "dolocationid",
            "dropoff_zone",
            "dropoff_borough",
        ]
    ].head()
)


**3.1.11** <font color = red>[3 marks]</font> <br>
Group data by location IDs to find the total number of trips per location ID

In [ ]:
trip_counts = (
    df.groupby("pulocationid")
    .size()
    .reset_index(name="trip_count")
    .sort_values("trip_count", ascending=False)
)

display(trip_counts.head(10))


**3.1.12** <font color = red>[2 marks]</font> <br>
Now, use the grouped data to add number of trips to the GeoDataFrame.

We will use this to plot a map of zones showing total trips per zone.

In [ ]:
zones_with_trips = zones.merge(trip_counts, left_on="locationid", right_on="pulocationid", how="left")
zones_with_trips["trip_count"] = zones_with_trips["trip_count"].fillna(0)

display(zones_with_trips[["locationid", "zone", "borough", "trip_count"]].head())


The next step is creating a color map (choropleth map) showing zones by the number of trips taken.

Again, you can use the `zones.plot()` method for this. [Plot Method GPD](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.plot.html#geopandas.GeoDataFrame.plot)

But first, you need to define the figure and axis for the plot.

`fig, ax = plt.subplots(1, 1, figsize = (12, 10))`

This function creates a figure (fig) and a single subplot (ax)

---

After setting up the figure and axis, we can proceed to plot the GeoDataFrame on this axis. This is done in the next step where we use the plot method of the GeoDataFrame.

You can define the following parameters in the `zones.plot()` method:
```
column = '',
ax = ax,
legend = True,
legend_kwds = {'label': "label", 'orientation': "<horizontal/vertical>"}
```

To display the plot, use `plt.show()`.

**3.1.13** <font color = red>[3 marks]</font> <br>
Plot a color-coded map showing zone-wise trips

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 12))
zones_with_trips.plot(
    column="trip_count",
    ax=ax,
    legend=True,
    cmap="YlOrRd",
    edgecolor="black",
    linewidth=0.2,
    legend_kwds={"label": "Number of sampled trips", "orientation": "vertical"},
)
ax.set_title("Zone-wise Taxi Trips in NYC")
ax.set_axis_off()
plt.show()


In [ ]:
display(
    zones_with_trips[["zone", "borough", "trip_count"]]
    .sort_values("trip_count", ascending=False)
    .head(15)
)


Here we have completed the temporal, financial and geographical analysis on the trip records.

**Compile your findings from general analysis below:**

You can consider the following points:

* Busiest hours, days and months
* Trends in revenue collected
* Trends in quarterly revenue
* How fare depends on trip distance, trip duration and passenger counts
* How tip amount depends on trip distance
* Busiest zones


#### **3.2** Detailed EDA: Insights and Strategies
<font color = red>[50 marks]</font> <br>

Having performed basic analyses for finding trends and patterns, we will now move on to some detailed analysis focussed on operational efficiency, pricing strategies, and customer experience.

##### Operational Efficiency

Analyze variations by time of day and location to identify bottlenecks or inefficiencies in routes

**3.2.1** <font color = red>[3 marks]</font> <br>
Identify slow routes by calculating the average time taken by cabs to get from one zone to another at different hours of the day.

Speed on a route *X* for hour *Y* = (*distance of the route X / average trip duration for hour Y*)

In [ ]:
route_hour_speed = (
    trip_zones[(trip_zones["trip_distance"] > 0) & (trip_zones["trip_duration_hr"] > 0)]
    .groupby(["pickup_zone", "dropoff_zone", "pickup_hour"], dropna=False)
    .agg(
        avg_distance=("trip_distance", "mean"),
        avg_duration_hr=("trip_duration_hr", "mean"),
        trip_count=("vendorid", "size"),
    )
    .reset_index()
)
route_hour_speed = route_hour_speed.dropna(subset=["pickup_zone", "dropoff_zone"])
route_hour_speed = route_hour_speed[route_hour_speed["trip_count"] >= 25].copy()
route_hour_speed["avg_speed_mph"] = route_hour_speed["avg_distance"] / route_hour_speed["avg_duration_hr"]
slow_routes = route_hour_speed.nsmallest(10, "avg_speed_mph")

display(slow_routes[["pickup_zone", "dropoff_zone", "pickup_hour", "trip_count", "avg_speed_mph"]])

plt.figure(figsize=(10, 5))
sns.barplot(
    data=slow_routes,
    y=slow_routes["pickup_zone"] + " -> " + slow_routes["dropoff_zone"],
    x="avg_speed_mph",
    color="#DD8452",
)
plt.title("Slowest High-Volume Routes by Hour")
plt.xlabel("Average speed (mph)")
plt.ylabel("Route")
plt.show()


How does identifying high-traffic, high-demand routes help us?

**3.2.2** <font color = red>[3 marks]</font> <br>
Calculate the number of trips at each hour of the day and visualise them. Find the busiest hour and show the number of trips for that hour.

In [ ]:
hourly_trip_counts = df.groupby("pickup_hour").size().reindex(range(24), fill_value=0)
busiest_hour = int(hourly_trip_counts.idxmax())

plt.figure(figsize=(12, 4))
sns.lineplot(x=hourly_trip_counts.index, y=hourly_trip_counts.values, marker="o")
plt.title("Trips by Hour of Day")
plt.xlabel("Pickup hour")
plt.ylabel("Number of sampled trips")
plt.xticks(range(24))
plt.show()

print(f"Busiest sampled hour: {busiest_hour}:00 with {hourly_trip_counts.max():,} trips")


Remember, we took a fraction of trips. To find the actual number, you have to scale the number up by the sampling ratio.

**3.2.3** <font color = red>[2 mark]</font> <br>
Find the actual number of trips in the five busiest hours

In [ ]:
sample_fraction = SAMPLE_FRACTION
actual_hourly_trips = (hourly_trip_counts / sample_fraction).round().astype(int)
top_five_actual_hours = actual_hourly_trips.sort_values(ascending=False).head(5)

display(top_five_actual_hours.to_frame("estimated_actual_trip_count"))


**3.2.4** <font color = red>[3 marks]</font> <br>
Compare hourly traffic pattern on weekdays. Also compare for weekend.

In [ ]:
daily_hourly_counts = (
    df.groupby(["pickup_date", "pickup_day_name", "pickup_hour"], observed=False)
    .size()
    .reset_index(name="trip_count")
)

weekday_profile = (
    daily_hourly_counts[~daily_hourly_counts["pickup_day_name"].isin(["Saturday", "Sunday"])]
    .groupby("pickup_hour")["trip_count"]
    .mean()
    .reindex(range(24), fill_value=0)
)
weekend_profile = (
    daily_hourly_counts[daily_hourly_counts["pickup_day_name"].isin(["Saturday", "Sunday"])]
    .groupby("pickup_hour")["trip_count"]
    .mean()
    .reindex(range(24), fill_value=0)
)

plt.figure(figsize=(12, 5))
sns.lineplot(x=weekday_profile.index, y=weekday_profile.values, label="Average weekday")
sns.lineplot(x=weekend_profile.index, y=weekend_profile.values, label="Average weekend")
plt.title("Hourly Traffic Pattern: Weekdays vs Weekends")
plt.xlabel("Pickup hour")
plt.ylabel("Average sampled trips")
plt.xticks(range(24))
plt.show()

daywise_hour_profile = (
    daily_hourly_counts.groupby(["pickup_day_name", "pickup_hour"], observed=False)["trip_count"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(12, 5))
sns.lineplot(data=daywise_hour_profile, x="pickup_hour", y="trip_count", hue="pickup_day_name")
plt.title("Average Hourly Traffic by Day of Week")
plt.xlabel("Pickup hour")
plt.ylabel("Average sampled trips")
plt.xticks(range(24))
plt.show()


What can you infer from the above patterns? How will finding busy and quiet hours for each day help us?

**3.2.5** <font color = red>[3 marks]</font> <br>
Identify top 10 zones with high hourly pickups. Do the same for hourly dropoffs. Show pickup and dropoff trends in these zones.

In [ ]:
top_pickup_zones = trip_zones["pickup_zone"].value_counts().head(10)
top_dropoff_zones = trip_zones["dropoff_zone"].value_counts().head(10)

display(top_pickup_zones.to_frame("pickup_count"))
display(top_dropoff_zones.to_frame("dropoff_count"))

pickup_hourly_top_zones = (
    trip_zones[trip_zones["pickup_zone"].isin(top_pickup_zones.index)]
    .groupby(["pickup_hour", "pickup_zone"])
    .size()
    .reset_index(name="trip_count")
)
dropoff_hourly_top_zones = (
    trip_zones[trip_zones["dropoff_zone"].isin(top_dropoff_zones.index)]
    .groupby(["pickup_hour", "dropoff_zone"])
    .size()
    .reset_index(name="trip_count")
)

plt.figure(figsize=(12, 5))
sns.lineplot(data=pickup_hourly_top_zones, x="pickup_hour", y="trip_count", hue="pickup_zone")
plt.title("Hourly Pickup Trends in Top 10 Pickup Zones")
plt.xlabel("Pickup hour")
plt.ylabel("Number of sampled trips")
plt.xticks(range(24))
plt.show()

plt.figure(figsize=(12, 5))
sns.lineplot(data=dropoff_hourly_top_zones, x="pickup_hour", y="trip_count", hue="dropoff_zone")
plt.title("Hourly Dropoff Trends in Top 10 Dropoff Zones")
plt.xlabel("Pickup hour")
plt.ylabel("Number of sampled trips")
plt.xticks(range(24))
plt.show()


**3.2.6** <font color = red>[3 marks]</font> <br>
Find the ratio of pickups and dropoffs in each zone. Display the 10 highest (pickup/drop) and 10 lowest (pickup/drop) ratios.

In [ ]:
pickup_zone_counts = trip_zones.groupby("pickup_zone").size().rename("pickup_count")
dropoff_zone_counts = trip_zones.groupby("dropoff_zone").size().rename("dropoff_count")
zone_flow_ratio = pd.concat([pickup_zone_counts, dropoff_zone_counts], axis=1).fillna(0)
zone_flow_ratio = zone_flow_ratio[(zone_flow_ratio["pickup_count"] > 0) & (zone_flow_ratio["dropoff_count"] > 0)].copy()
zone_flow_ratio["pickup_drop_ratio"] = zone_flow_ratio["pickup_count"] / zone_flow_ratio["dropoff_count"]

print("Highest pickup/dropoff ratios:")
display(zone_flow_ratio.sort_values("pickup_drop_ratio", ascending=False).head(10))
print("Lowest pickup/dropoff ratios:")
display(zone_flow_ratio.sort_values("pickup_drop_ratio", ascending=True).head(10))


**3.2.7** <font color = red>[3 marks]</font> <br>
Identify zones with high pickup and dropoff traffic during night hours (11PM to 5AM)

In [ ]:
night_trips = trip_zones[trip_zones["pickup_hour"].isin(NIGHT_HOURS)].copy()
night_pickup_zones = night_trips["pickup_zone"].value_counts().head(10)
night_dropoff_zones = night_trips["dropoff_zone"].value_counts().head(10)

print("Top 10 night pickup zones:")
display(night_pickup_zones.to_frame("pickup_count"))
print("Top 10 night dropoff zones:")
display(night_dropoff_zones.to_frame("dropoff_count"))


Now, let us find the revenue share for the night time hours and the day time hours. After this, we will move to deciding a pricing strategy.

**3.2.8** <font color = red>[2 marks]</font> <br>
Find the revenue share for nighttime and daytime hours.

In [ ]:
df["time_period"] = np.where(df["pickup_hour"].isin(NIGHT_HOURS), "Night", "Day")
revenue_share_by_period = df.groupby("time_period")["total_amount"].sum().sort_values(ascending=False)
revenue_share_percent = (revenue_share_by_period / revenue_share_by_period.sum() * 100).round(2)

plt.figure(figsize=(6, 4))
sns.barplot(x=revenue_share_percent.index, y=revenue_share_percent.values, palette="mako")
plt.title("Revenue Share: Day vs Night")
plt.xlabel("Time period")
plt.ylabel("Revenue share (%)")
plt.show()

display(revenue_share_percent.to_frame("revenue_share_percent"))


##### Pricing Strategy

**3.2.9** <font color = red>[2 marks]</font> <br>
For the different passenger counts, find the average fare per mile per passenger.

For instance, suppose the average fare per mile for trips with 3 passengers is 3 USD/mile, then the fare per mile per passenger will be 1 USD/mile.

In [ ]:
fare_per_mile_df = revenue_df[revenue_df["passenger_count"] > 0].copy()
fare_per_mile_df["fare_per_mile"] = fare_per_mile_df["fare_amount"] / fare_per_mile_df["trip_distance"]
fare_per_mile_df["fare_per_mile_per_passenger"] = (
    fare_per_mile_df["fare_per_mile"] / fare_per_mile_df["passenger_count"]
)

fare_per_mile_passenger_summary = (
    fare_per_mile_df.groupby("passenger_count")["fare_per_mile_per_passenger"]
    .mean()
    .sort_index()
)

plt.figure(figsize=(8, 4))
sns.barplot(
    x=fare_per_mile_passenger_summary.index,
    y=fare_per_mile_passenger_summary.values,
    color="#64B5CD",
)
plt.title("Average Fare per Mile per Passenger")
plt.xlabel("Passenger count")
plt.ylabel("Average fare per mile per passenger")
plt.show()

display(fare_per_mile_passenger_summary.to_frame("avg_fare_per_mile_per_passenger"))


**3.2.10** <font color = red>[3 marks]</font> <br>
Find the average fare per mile by hours of the day and by days of the week

In [ ]:
hourly_fare_per_mile = fare_per_mile_df.groupby("pickup_hour")["fare_per_mile"].mean().reindex(range(24))
daily_fare_per_mile = (
    fare_per_mile_df.groupby("pickup_day_name", observed=False)["fare_per_mile"]
    .mean()
    .reindex(DAY_ORDER)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
sns.lineplot(x=hourly_fare_per_mile.index, y=hourly_fare_per_mile.values, marker="o", ax=axes[0])
axes[0].set_title("Average Fare per Mile by Hour")
axes[0].set_xlabel("Pickup hour")
axes[0].set_ylabel("Fare per mile")
axes[0].set_xticks(range(24))

sns.barplot(x=daily_fare_per_mile.index, y=daily_fare_per_mile.values, ax=axes[1], palette="flare")
axes[1].set_title("Average Fare per Mile by Day of Week")
axes[1].set_xlabel("Day of week")
axes[1].set_ylabel("Fare per mile")
axes[1].tick_params(axis="x", rotation=30)
plt.show()

display(hourly_fare_per_mile.to_frame("avg_fare_per_mile"))
display(daily_fare_per_mile.to_frame("avg_fare_per_mile"))


**3.2.11** <font color = red>[3 marks]</font> <br>
Analyse the average fare per mile for the different vendors for different hours of the day

In [ ]:
vendor_hourly_fare = (
    fare_per_mile_df.groupby(["vendor_name", "pickup_hour"])["fare_per_mile"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(12, 5))
sns.lineplot(data=vendor_hourly_fare, x="pickup_hour", y="fare_per_mile", hue="vendor_name", marker="o")
plt.title("Average Fare per Mile by Vendor and Hour")
plt.xlabel("Pickup hour")
plt.ylabel("Fare per mile")
plt.xticks(range(24))
plt.show()

display(vendor_hourly_fare.head(20))


**3.2.12** <font color = red>[5 marks]</font> <br>
Compare the fare rates of the different vendors in a tiered fashion. Analyse the average fare per mile for distances upto 2 miles. Analyse the fare per mile for distances from 2 to 5 miles. And then for distances more than 5 miles.


In [ ]:
vendor_distance_tiers = fare_per_mile_df.copy()
vendor_distance_tiers["distance_tier"] = pd.cut(
    vendor_distance_tiers["trip_distance"],
    bins=[0, 2, 5, np.inf],
    labels=["Up to 2 miles", "2 to 5 miles", "More than 5 miles"],
    include_lowest=True,
)

tiered_vendor_fares = (
    vendor_distance_tiers.groupby(["distance_tier", "vendor_name"], observed=False)["fare_per_mile"]
    .mean()
    .reset_index()
)

display(tiered_vendor_fares.pivot(index="distance_tier", columns="vendor_name", values="fare_per_mile"))

plt.figure(figsize=(10, 5))
sns.barplot(data=tiered_vendor_fares, x="distance_tier", y="fare_per_mile", hue="vendor_name")
plt.title("Vendor Fare per Mile Across Distance Tiers")
plt.xlabel("Distance tier")
plt.ylabel("Average fare per mile")
plt.show()


##### Customer Experience and Other Factors

**3.2.13** <font color = red>[5 marks]</font> <br>
Analyse average tip percentages based on trip distances, passenger counts and time of pickup. What factors lead to low tip percentages?

In [ ]:
tip_percentage_df = tip_analysis_df.copy()
tip_percentage_df["tip_percentage"] = (
    tip_percentage_df["tip_amount"] / tip_percentage_df["fare_amount"] * 100
)
tip_percentage_df = tip_percentage_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["tip_percentage"])
tip_percentage_df["distance_tier"] = pd.cut(
    tip_percentage_df["trip_distance"],
    bins=[0, 2, 5, 10, np.inf],
    labels=["Up to 2", "2 to 5", "5 to 10", "10+"],
    include_lowest=True,
)

tip_by_distance_tier = tip_percentage_df.groupby("distance_tier", observed=False)["tip_percentage"].mean()
tip_by_passenger_count = tip_percentage_df.groupby("passenger_count")["tip_percentage"].mean().sort_index()
tip_by_hour = tip_percentage_df.groupby("pickup_hour")["tip_percentage"].mean().reindex(range(24))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.barplot(x=tip_by_distance_tier.index, y=tip_by_distance_tier.values, ax=axes[0], color="#55A868")
axes[0].set_title("Tip % by Distance Tier")
axes[0].set_xlabel("Distance tier")
axes[0].set_ylabel("Average tip %")

sns.barplot(x=tip_by_passenger_count.index, y=tip_by_passenger_count.values, ax=axes[1], color="#C44E52")
axes[1].set_title("Tip % by Passenger Count")
axes[1].set_xlabel("Passenger count")
axes[1].set_ylabel("Average tip %")

sns.lineplot(x=tip_by_hour.index, y=tip_by_hour.values, ax=axes[2], marker="o")
axes[2].set_title("Tip % by Pickup Hour")
axes[2].set_xlabel("Pickup hour")
axes[2].set_ylabel("Average tip %")
axes[2].set_xticks(range(24))
plt.show()

print("Lowest tip-percentage hours:")
display(tip_by_hour.sort_values().head(5).to_frame("avg_tip_percentage"))


Additional analysis [optional]: Let's try comparing cases of low tips with cases of high tips to find out if we find a clear aspect that drives up the tipping behaviours

In [ ]:
low_tip_trips = tip_percentage_df[tip_percentage_df["tip_percentage"] < 10].copy()
high_tip_trips = tip_percentage_df[tip_percentage_df["tip_percentage"] > 25].copy()

comparison_summary = pd.DataFrame(
    {
        "metric": [
            "average_trip_distance",
            "average_fare_amount",
            "average_passenger_count",
            "average_pickup_hour",
            "credit_card_share_percent",
            "cash_share_percent",
        ],
        "low_tip_group": [
            low_tip_trips["trip_distance"].mean(),
            low_tip_trips["fare_amount"].mean(),
            low_tip_trips["passenger_count"].mean(),
            low_tip_trips["pickup_hour"].mean(),
            (low_tip_trips["payment_type_name"].eq("Credit card").mean() * 100),
            (low_tip_trips["payment_type_name"].eq("Cash").mean() * 100),
        ],
        "high_tip_group": [
            high_tip_trips["trip_distance"].mean(),
            high_tip_trips["fare_amount"].mean(),
            high_tip_trips["passenger_count"].mean(),
            high_tip_trips["pickup_hour"].mean(),
            (high_tip_trips["payment_type_name"].eq("Credit card").mean() * 100),
            (high_tip_trips["payment_type_name"].eq("Cash").mean() * 100),
        ],
    }
)

display(comparison_summary)


**3.2.14** <font color = red>[3 marks]</font> <br>
Analyse the variation of passenger count across hours and days of the week.

In [ ]:
avg_passenger_by_hour = df.groupby("pickup_hour")["passenger_count"].mean().reindex(range(24), fill_value=0)
avg_passenger_by_day = (
    df.groupby("pickup_day_name", observed=False)["passenger_count"]
    .mean()
    .reindex(DAY_ORDER)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
sns.lineplot(x=avg_passenger_by_hour.index, y=avg_passenger_by_hour.values, marker="o", ax=axes[0])
axes[0].set_title("Average Passenger Count by Hour")
axes[0].set_xlabel("Pickup hour")
axes[0].set_ylabel("Average passenger count")
axes[0].set_xticks(range(24))

sns.barplot(x=avg_passenger_by_day.index, y=avg_passenger_by_day.values, ax=axes[1], palette="coolwarm")
axes[1].set_title("Average Passenger Count by Day of Week")
axes[1].set_xlabel("Day of week")
axes[1].set_ylabel("Average passenger count")
axes[1].tick_params(axis="x", rotation=30)
plt.show()


**3.2.15** <font color = red>[2 marks]</font> <br>
Analyse the variation of passenger counts across zones

In [ ]:
avg_passenger_by_zone = (
    trip_zones.groupby("pickup_zone")["passenger_count"]
    .mean()
    .dropna()
    .sort_values(ascending=False)
    .head(15)
)

display(avg_passenger_by_zone.to_frame("avg_passenger_count"))

plt.figure(figsize=(10, 6))
sns.barplot(x=avg_passenger_by_zone.values, y=avg_passenger_by_zone.index, color="#937860")
plt.title("Zones with Highest Average Passenger Count")
plt.xlabel("Average passenger count")
plt.ylabel("Pickup zone")
plt.show()


In [ ]:
avg_passenger_by_zone_id = (
    df.groupby("pulocationid")["passenger_count"]
    .mean()
    .reset_index(name="avg_passenger_count")
)

zones_with_trips = zones_with_trips.drop(columns=["avg_passenger_count"], errors="ignore")
zones_with_trips = zones_with_trips.merge(
    avg_passenger_by_zone_id,
    left_on="locationid",
    right_on="pulocationid",
    how="left",
)
zones_with_trips["avg_passenger_count"] = zones_with_trips["avg_passenger_count"].fillna(0)

display(
    zones_with_trips[["zone", "borough", "trip_count", "avg_passenger_count"]]
    .sort_values("avg_passenger_count", ascending=False)
    .head(10)
)


Find out how often surcharges/extra charges are applied to understand their prevalance

**3.2.16** <font color = red>[5 marks]</font> <br>
Analyse the pickup/dropoff zones or times when extra charges are applied more frequently

In [ ]:
surcharge_columns = [
    column for column in [
        "extra",
        "mta_tax",
        "improvement_surcharge",
        "congestion_surcharge",
        "airport_fee",
        "tolls_amount",
    ]
    if column in df.columns
]

surcharge_prevalence = (
    pd.Series(
        {column: (df[column] > 0).mean() * 100 for column in surcharge_columns},
        name="percent_of_trips",
    )
    .sort_values(ascending=False)
)

display(surcharge_prevalence.to_frame())

def any_positive_charge(dataframe):
    return dataframe[surcharge_columns].gt(0).any(axis=1).astype(int)

trip_zones["any_extra_charge"] = any_positive_charge(trip_zones)
hourly_extra_charge = trip_zones.groupby("pickup_hour")["any_extra_charge"].mean().mul(100)
top_pickup_zones_extra = (
    trip_zones.groupby("pickup_zone")["any_extra_charge"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .head(10)
)
top_dropoff_zones_extra = (
    trip_zones.groupby("dropoff_zone")["any_extra_charge"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .head(10)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.lineplot(x=hourly_extra_charge.index, y=hourly_extra_charge.values, marker="o", ax=axes[0])
axes[0].set_title("Trips with Extra Charges by Hour")
axes[0].set_xlabel("Pickup hour")
axes[0].set_ylabel("Trips with extra charges (%)")
axes[0].set_xticks(range(24))

sns.barplot(x=top_pickup_zones_extra.values, y=top_pickup_zones_extra.index, ax=axes[1], color="#DA8BC3")
axes[1].set_title("Top Pickup Zones by Extra-Charge Frequency")
axes[1].set_xlabel("Trips with extra charges (%)")
axes[1].set_ylabel("Pickup zone")
plt.show()

print("Top dropoff zones by extra-charge frequency:")
display(top_dropoff_zones_extra.to_frame("extra_charge_percent"))


## **4** Conclusion
<font color = red>[15 marks]</font> <br>

### **4.1** Final Insights and Recommendations
<font color = red>[15 marks]</font> <br>

Conclude your analyses here. Include all the outcomes you found based on the analysis.

Based on the insights, frame a concluding story explaining suitable parameters such as location, time of the day, day of the week etc. to be kept in mind while devising a strategy to meet customer demand and optimise supply.

**4.1.1** <font color = red>[5 marks]</font> <br>
Recommendations to optimize routing and dispatching based on demand patterns and operational inefficiencies

In [ ]:
slowest_route_text = "route information not available"
if not slow_routes.empty:
    slowest_route = slow_routes.iloc[0]
    slowest_route_text = (
        f"{slowest_route['pickup_zone']} -> {slowest_route['dropoff_zone']} at {int(slowest_route['pickup_hour'])}:00 "
        f"({slowest_route['avg_speed_mph']:.2f} mph)"
    )

recommendation_md = f"""
**Routing and dispatching recommendations**

- Match driver rosters to the strongest demand window around **{busiest_hour}:00**, because that is the busiest sampled hour in the 2023 data.
- Use the weekday and weekend hourly curves to stagger shift starts instead of running a flat schedule through the day.
- Monitor slow but high-volume routes such as **{slowest_route_text}** for congestion-aware dispatching and realistic ETA promises.
- Route drivers toward high-volume pickup corridors before peaks so fewer cabs are idle in low-demand periods.
"""

display(Markdown(recommendation_md))


**4.1.2** <font color = red>[5 marks]</font> <br>

Suggestions on strategically positioning cabs across different zones to make best use of insights uncovered by analysing trip trends across time, days and months.

In [ ]:
primary_pickup_zone = top_pickup_zones.index[0] if len(top_pickup_zones) else "the leading pickup zone"
primary_night_zone = night_pickup_zones.index[0] if len(night_pickup_zones) else "the leading night zone"

positioning_md = f"""
**Strategic positioning suggestions**

- Keep a larger share of cabs positioned near **{primary_pickup_zone}** and the other top pickup zones identified in the zone analysis.
- Maintain a night-focused pool of drivers around **{primary_night_zone}** and nearby night-demand clusters from 11 PM to 5 AM.
- Rebalance cabs away from zones with persistently low pickup/dropoff ratios and toward areas where pickups consistently exceed dropoffs.
- Use the choropleth map and hourly zone trends together to plan borough-level rebalancing between peaks.
"""

display(Markdown(positioning_md))


**4.1.3** <font color = red>[5 marks]</font> <br>
Propose data-driven adjustments to the pricing strategy to maximize revenue while maintaining competitive rates with other vendors.

In [ ]:
highest_fare_hour = int(hourly_fare_per_mile.idxmax())
lowest_fare_hour = int(hourly_fare_per_mile.idxmin())
night_revenue_share = revenue_share_percent.get("Night", 0)

tiered_pricing_md = f"""
**Pricing strategy recommendations**

- Protect pricing power during high-yield hours such as **{highest_fare_hour}:00**, while using quieter periods like **{lowest_fare_hour}:00** for targeted promotions or loyalty offers.
- Maintain close vendor benchmarking by distance tier, especially for trips up to 2 miles where fare-per-mile differences are most visible to passengers.
- Since night trips contribute about **{night_revenue_share:.2f}%** of sampled revenue, night operations can justify careful premium positioning if service reliability remains strong.
- Use tip-percentage and passenger-count trends to avoid blanket price increases on trip types where customer willingness to pay appears weaker.
"""

display(Markdown(tiered_pricing_md))
